In [ ]:
from flask import Flask, make_response,session
from datetime import datetime, timedelta
import os
app = Flask(__name__)
app.secret_key = "123123"

# 1.2 Cookie 和 Session

## Cookie
Cookie 通过在客户端记录信息来记录客户身份，做到服务的保持<br>
通过在客户端存储一个小文本文件来实现，里面存储了一个 Session ID，服务器通过这个 Session ID 来识别用户并获取对应的 Session 数据。<br>
**Cookie 是由服务器生成的，发送给客户端并保存到客户端。** 后者会在之后的请求中携带这个 Cookie，服务器通过解析 Cookie 来识别用户。<br>

> 可以理解为，你的每次登陆都会创建一个专属的保险箱（Session），服务器把钥匙（Session ID）放在你口袋（Cookie）里。<br>
访问时，你出示钥匙，服务器就能打开对应的保险箱读取你的信息。<br>
这个保险箱装着你的所有东西，宏观如用户名、密码；微观如购物车、“我的足迹”等。<br>
当你登出或者一段时间没有登陆时，服务器就会销毁这个保险箱，但在此类情况之外的时候，你只要进网站，就可以通过钥匙打开保险箱，继续使用里面的东西。<br>
这就是我们常见的“保持登录状态”的原理。<br>
具体能保持多久，取决于服务器设置的 Session 过期时间。你的 Cookie 单纯是一个“钥匙”。如果 Session 过期或毁约，Cookie 也就没有作用了。<br>

- 使用 `set_cookie()` 方法可以设置 Cookie 值；
- 使用 `get_cookie()` 方法可以获取 Cookie。<br>
- 使用 `delete_cookie()` 方法可以删除单个 Cookie。<br>
- 使用 `clear_cookie()` 方法可以删除所有 Cookie。<br>

也可以通过 `headers['Set-Cookie']` 来设置 Cookie。<br>

cookie 是 Flask 自带的，无需特别导入。<br>

### Cookie 的持续时间

默认情况下，Cookie 是会话级别的，也就是说，当浏览器关闭时，Cookie 就会被删除。<br>
如果需要让 Cookie 持续更长时间，可以设置 `max_age` 或 `expires` 参数。<br>
- `max_age` 是以秒为单位的持续时间，例如 `max_age=3600` 表示 Cookie 将持续 1 小时。
- `expires` 是一个具体的过期时间，可以使用 `datetime` 模块来设置，<br>例如 `expires=datetime.datetime(2024, 12, 31)` 表示 Cookie 将在 2024 年 12 月 31 日过期。<br>


In [ ]:
@app.route('/setcookie')
def setcookie():
    rescookie = make_response('A cookie set in this page')
    rescookie.set_cookie('name01', 'pylor')
    # rescookie.set_cookie('name02', 'pylorSun')
    # rescookie.set_cookie('name03', 'PylorSun', expires=datetime(2026,3,20))		# 使用 expires 特定过期时间
    # rescookie.set_cookie('name04', 'PylorRealvent', max_age=3600)					# 使用 max_age 设定保质期
    # rescookie.headers["Set-Cookie"] = "anotherName=PylorHeader;max-age=500"		# 使用 headers["Set-Cookie"] 创建 cookie
    return rescookie

if __name__ == '__main__':
    app.run()

'''
登陆 127.0.0.1:5000/setcookie 
一个响应对象中，可以有多个 cookie。一个 set_cookie 可以创建一个 cookie。
虽然 Flask 在设计上没有限制 cookie 的创建上限，但浏览器对每个域名会限制 cookie，通常是 50 - 150 个。
浏览器如果整体未关闭，cookie 连接就会一直存在。
'''

当然，也可以搭配 [Request 和 Response](1_1_Request_n_Response.ipynb) 中的传入变量内容进行一些操作。<br>

## Session
通过在服务器端记录信息来记录客户身份，做到保持。<br>
Session 是基于 Cookie 实现的，Session ID 存储在 Cookie 中，服务器通过 Session ID 来识别用户并获取对应的 Session 数据。<br>

Session 的使用需要在 **开始位置** 设置密钥。一般使用这两种方式：

- `app.secret_key = 'your_secret_key'`
- `app.config['SECRET_KEY'] = 'your_secret_key'`

前者直接设置属性，后者通过配置字典设置。<br>
它们可以是一个自定义的任意字符串，但在生产环境中，建议使用一个随机生成的字符串来增强安全性。<br>
如 `os.urandom()` 。<br>

> `urandom` 来自 os 模块，用于生成真随机数。<br>
与函数生成的“伪随机数”不同，它通过操作系统传递的噪声（如鼠标移动、键盘输入等）来生成随机数，因此具有较高的随机性和安全性。

当然，Session 也支持以下方法：
- `session['key'] = value`：设置 Session 数据。
- `session.get('key')`：获取 Session 数据。<br><br>
- `session.pop('key')`：删除 Session 数据。 此时删除并返回该值，不存在则抛出 KeyError。<br>
- `del session['key']`：另一种相似删除方式，不存在时抛出 KeyError。<br>
	或者 `session.pop('key', None)`，如果 key 不存在则返回 None。<br>
	抑或 `session['key'] = False`，将 key 的值设置为 False 来表示删除。<br>
- `session.clear()`：清除所有 Session 数据。

In [ ]:
@app.route('/setsession')
def setsession():
    session['name'] = 'Pylor'
    session.permanent = True
    return '这个页面通过 session[] 的词典方式设置了 session'

@app.route('/getsession')
def getsession():
    # 所有人都会建议在 get() 时加上一个若否的 fallback 值，来避免 KeyError
    # 这也符合 Flask 不能返回 None 的类型检查要求。
    return session.get('name', 'No name in session')

@app.route('/delsession')
def delsession():
    session.pop('name', None)
    return '这个页面通过 session.pop() 的方式删除了 session'

if __name__ == '__main__':
    app.run()
    
'''
登陆以上地址，会在开发者页面看到对应的变化。
'''